# Data Vault Workflow: From Raw Data to Quantum-Safe Storage

This notebook walks through the **complete data protection lifecycle** that a data analyst
would follow when working with sensitive records every day:

```
Fetch  →  Scan  →  Anonymize  →  Encrypt  →  Store  →  Reopen  →  Self-Destruct
```

**Use case:** You pull customer records from a SQL database each morning. The data contains
Norwegian national IDs (fødselsnummer), health diagnoses, bank account numbers, and salaries.
Under GDPR and DORA you must protect this data at rest, limit retention, and be able to
prove what protections were applied.

Zipminator provides:

| Capability | What it does |
|---|---|
| **PII Scanner** | Detects personally identifiable information across 15 country patterns |
| **Anonymization Engine** | 10 levels from SHA-256 hashing to differential privacy |
| **Zipndel** | Encrypts + compresses a DataFrame into a password-protected or PQC-secured ZIP |
| **Unzipndel** | Decrypts and restores the original DataFrame |
| **PQC Vault** | Kyber768 (ML-KEM-768) key encapsulation for quantum-resistant protection |
| **Self-Destruct** | DoD 5220.22-M 3-pass secure overwrite with configurable timers |

**Prerequisites:** `pip install zipminator[all]` or `uv pip install zipminator[all]`

In [ ]:
import pandas as pd
import numpy as np
import os
import tempfile
import shutil

# Plotly visualization helpers
import sys; sys.path.insert(0, "..")
from _helpers.viz import *
import plotly.graph_objects as go

# Create a temporary working directory for all file operations
WORK_DIR = tempfile.mkdtemp(prefix="zipminator_vault_")
print(f"Working directory: {WORK_DIR}")

# Simulate fetching from a SQL database
df = pd.DataFrame({
    'customer_id': [1001, 1002, 1003, 1004, 1005],
    'full_name': ['Ola Nordmann', 'Kari Hansen', 'Erik Johansen', 'Liv Andersen', 'Per Nilsen'],
    'email': ['ola@example.no', 'kari@bank.no', 'erik@hospital.no', 'liv@gov.no', 'per@school.no'],
    'fodselsnummer': ['12345678901', '98765432109', '45678901234', '67890123456', '23456789012'],
    'salary_nok': [650000, 820000, 540000, 710000, 480000],
    'diagnosis': ['Hypertension', 'Diabetes Type 2', 'None', 'Asthma', 'None'],
    'account_no': ['1234.56.78901', '9876.54.32109', '4567.89.01234', '6789.01.23456', '2345.67.89012'],
})

print(f"Fetched {len(df)} records from database")
print(df.to_string(index=False))

## Step 1: PII Scanning

**PII** (Personally Identifiable Information) is any data that can identify a specific person,
either directly or in combination with other data.

Examples:
- **Direct identifiers:** Name, email, national ID (fødselsnummer), phone number
- **Quasi-identifiers:** Salary (combined with employer = identity), diagnosis (combined with date = identity)
- **Safe data:** Aggregated statistics, anonymized IDs, category labels

The scanner checks each column against patterns for 15 countries (NO, SE, DK, FI, DE, FR, UK, US, UAE, IN, BR, JP, CA, AU, and EU-wide patterns).
It returns a risk assessment and recommended protection level.

In [ ]:
from zipminator.crypto.pii_scanner import PIIScanner

scanner = PIIScanner()
results = scanner.scan_dataframe(df)

print("=" * 60)
print("PII SCAN RESULTS")
print("=" * 60)
print(f"PII detected: {results['pii_detected']}")
print(f"Risk level: {results['risk_level']}")
print(f"Recommended anonymization level: {results['recommended_anonymization_level']}")
print()

print("Column-level findings:")
print("-" * 60)
for col_name, pii_types in results.get('columns_with_pii', {}).items():
    print(f"  {col_name:20s} -> {pii_types}")

print()
print("Detailed matches:")
print("-" * 60)
for match in results.get('matches', []):
    print(f"  {match.pii_type.value:25s} in '{match.column}' (confidence: {match.confidence:.0%})")

print()
print("Full summary:")
print(results['summary'])

In [ ]:
# ── PII Type Distribution in Sample Dataset ──
pii_types = ["Email", "Norwegian FNR", "Norwegian Account", "Phone"]
pii_counts = [1, 1, 2, 1]

fig = zm_pie(
    labels=pii_types,
    values=pii_counts,
    title="PII Type Distribution in Sample Dataset",
    hole=0.45,
)
fig.update_traces(
    textinfo="label+percent+value",
    texttemplate="%{label}<br>%{value} col(s) (%{percent})",
)
fig.update_layout(height=400)
fig.show()

## Step 2: Choose Your Protection

Zipminator offers two encryption paths:

| Method | Best for | Security level |
|---|---|---|
| **Password (AES-256)** | Day-to-day work, sharing with colleagues | Strong against classical computers |
| **PQC Vault (Kyber768 / ML-KEM-768)** | Long-term archives, regulatory compliance | Resistant to both classical and quantum computers |

Both methods compress the data before encryption, reducing file size and eliminating plaintext on disk.

### Option A: Password-Protected Storage

In [ ]:
from zipminator.crypto.zipit import Zipndel

vault_path = os.path.join(WORK_DIR, 'customers_daily')

znd = Zipndel(
    file_name=vault_path,
    file_format='csv',
    password='demo-password-2026',
    compliance_check=True,
    audit_trail=True,
)

znd.zipit(df)

zip_file = f"{vault_path}.zip"
if os.path.exists(zip_file):
    size_kb = os.path.getsize(zip_file) / 1024
    print(f"Encrypted vault created: {os.path.basename(zip_file)}")
    print(f"Size: {size_kb:.1f} KB (compressed + AES-256 encrypted)")
else:
    print("Vault file not found - check Zipndel output above for errors")

**What just happened:**

1. The DataFrame was serialized to CSV in memory
2. A PII compliance scan ran automatically (`compliance_check=True`)
3. The CSV was compressed and encrypted with AES-256 using your password
4. The encrypted `.zip` file was written to disk
5. No plaintext CSV exists on disk; only the encrypted archive

The original data never touches the filesystem in cleartext.

In [ ]:
from zipminator import keypair, encapsulate, decapsulate
from zipminator.crypto.quantum_random import QuantumRandom

qr = QuantumRandom()
seed = qr.randbytes(32)
print(f"Entropy seed: {seed.hex()[:32]}... ({len(seed)} bytes)")
print(f"Source: {'quantum pool' if qr.get_stats().get('pool_available') else 'OS entropy (classical)'}")

pk, sk = keypair()
pk_bytes = pk.to_bytes()
sk_bytes = sk.to_bytes()
print(f"Public key:  {len(pk_bytes):,} bytes (share with anyone)")
print(f"Secret key:  {len(sk_bytes):,} bytes (keep safe!)")

ct, shared_secret_enc = encapsulate(pk)
shared_secret_dec = decapsulate(ct, sk)

print(f"\nCiphertext:    {len(ct.to_bytes()):,} bytes")
print(f"Shared secret: {shared_secret_enc.hex()[:32]}... ({len(shared_secret_enc)} bytes)")
print(f"Secrets match: {shared_secret_enc == shared_secret_dec}")

pk_path = os.path.join(WORK_DIR, 'vault_public.key')
sk_path = os.path.join(WORK_DIR, 'vault_secret.key')
with open(pk_path, 'wb') as f:
    f.write(pk_bytes)
with open(sk_path, 'wb') as f:
    f.write(sk_bytes)
print(f"\nKeys saved to: {os.path.basename(pk_path)}, {os.path.basename(sk_path)}")
print("In production: store the secret key in HSM or OS keychain.")

In [ ]:
# ── Encryption Strength Indicator: ML-KEM-768 ──
fig = zm_gauge(
    value=192,
    title="ML-KEM-768 Security Level",
    max_val=256,
    suffix=" bits",
)
fig.update_layout(
    annotations=[dict(
        text="NIST FIPS 203 | Category 3 Security",
        x=0.5, y=-0.05, showarrow=False,
        font=dict(size=13, color=ZM_COLORS["emerald"]),
        xref="paper", yref="paper",
    )],
    height=350,
)
fig.show()

## Step 3: Auto-Anonymization Before Storage

Even encrypted data benefits from anonymization as defense-in-depth:
if the encryption is ever broken, the underlying data is still protected.

Zipndel supports two column-level transformations:

| Parameter | Transformation | Reversible? |
|---|---|---|
| `mask_columns` | SHA-256 hash (one-way) | No |
| `anonymize_columns` | Random replacement | No |

For reversible pseudonymization, see Step 5 below.

In [ ]:
anon_vault_path = os.path.join(WORK_DIR, 'customers_anonymized')

znd_anon = Zipndel(
    file_name=anon_vault_path,
    file_format='csv',
    password='demo-password-2026',
    compliance_check=True,
    anonymization_level=1,
    mask_columns=['fodselsnummer'],
    anonymize_columns=['full_name'],
    audit_trail=True,
)

znd_anon.zipit(df)

anon_zip = f"{anon_vault_path}.zip"
if os.path.exists(anon_zip):
    size_kb = os.path.getsize(anon_zip) / 1024
    print(f"\nAnonymized vault created: {os.path.basename(anon_zip)} ({size_kb:.1f} KB)")
    print("\nWhat was transformed before encryption:")
    print("  fodselsnummer -> SHA-256 hash (irreversible, for linkage analysis only)")
    print("  full_name     -> random replacement (original names destroyed)")
    print("  email, salary, diagnosis, account_no -> encrypted but not anonymized")

In [ ]:
# ── Animated Bar: Vault Workflow Pipeline ──
steps = ["Raw Data", "PII Scanned", "Anonymized", "Encrypted", "Stored"]
raw_size, scan_size, anon_size, enc_size, stored_size = 443, 463, 520, 640, 580

frames_data = [
    {"name": "1. Raw Data",
     "x": steps, "y": [raw_size, 0, 0, 0, 0],
     "color": [ZM_COLORS["rose"], "#1e293b", "#1e293b", "#1e293b", "#1e293b"]},
    {"name": "2. PII Scanned",
     "x": steps, "y": [raw_size, scan_size, 0, 0, 0],
     "color": [ZM_COLORS["rose"], ZM_COLORS["amber"], "#1e293b", "#1e293b", "#1e293b"]},
    {"name": "3. Anonymized",
     "x": steps, "y": [raw_size, scan_size, anon_size, 0, 0],
     "color": [ZM_COLORS["rose"], ZM_COLORS["amber"], ZM_COLORS["violet"], "#1e293b", "#1e293b"]},
    {"name": "4. Encrypted",
     "x": steps, "y": [raw_size, scan_size, anon_size, enc_size, 0],
     "color": [ZM_COLORS["rose"], ZM_COLORS["amber"], ZM_COLORS["violet"], ZM_COLORS["cyan"], "#1e293b"]},
    {"name": "5. Stored",
     "x": steps, "y": [raw_size, scan_size, anon_size, enc_size, stored_size],
     "color": [ZM_COLORS["rose"], ZM_COLORS["amber"], ZM_COLORS["violet"], ZM_COLORS["cyan"], ZM_COLORS["emerald"]]},
]

fig = zm_animated_bar(
    frames_data,
    title="Vault Workflow: Data Transformation Pipeline",
    x_label="Pipeline Stage",
    y_label="Data Size (bytes)",
)
fig.update_layout(height=450)
fig.show()

In [ ]:
# ── Operation Latency: Stacked Bar by Algorithm ──
fig = zm_stacked_bar(
    x=["Encrypt", "Decrypt", "Key Gen", "Sign", "Verify"],
    y_dict={
        "ML-KEM-768":  [0.3, 0.2, 0.5, 0.4, 0.3],
        "RSA-2048":    [1.2, 0.8, 2.1, 1.5, 1.0],
        "AES-256-GCM": [0.1, 0.1, 0.05, None, None],
    },
    title="Operation Latency (ms) — PQC vs Classical",
)
fig.show()

## Step 4: Reopening the Next Day

The next morning you need the data again. `Unzipndel` decrypts and decompresses
the vault back into a DataFrame.

**Note:** `Unzipndel.unzipit()` prompts for the password via `getpass` (no echo),
which does not work in non-interactive notebook environments. Below we show the
standard usage pattern and then demonstrate direct extraction with `pyzipper`.

In [ ]:
# --- Standard usage (interactive terminal) ---
# from zipminator.crypto.unzipit import Unzipndel
# uznd = Unzipndel(file_name='customers_daily', file_format='csv')
# df_restored = uznd.unzipit()  # prompts for password via getpass

# --- Notebook-compatible demonstration ---
import pyzipper
import io

zip_file = f"{vault_path}.zip"
password = b'demo-password-2026'

with pyzipper.AESZipFile(zip_file) as zf:
    zf.setpassword(password)
    print("Archive contents:")
    for info in zf.infolist():
        print(f"  {info.filename} ({info.file_size:,} bytes uncompressed)")

    data_name = zf.namelist()[0]
    csv_data = zf.read(data_name)
    df_restored = pd.read_csv(io.BytesIO(csv_data))

print(f"\nRestored {len(df_restored)} records:")
print(df_restored.to_string(index=False))

match = df.reset_index(drop=True).equals(df_restored.reset_index(drop=True))
print(f"\nRound-trip integrity check: {'PASS' if match else 'FAIL'}")

## Step 5: Reversible Pseudonymization (L3 Tokenization)

Sometimes you need to share data with analysts who should not see real values,
but you need to reverse the transformation later (e.g., for audit or linking records).

The `AnonymizationEngine` at **level 3** produces deterministic tokens:
the same input always produces the same token, and the token map allows reversal.

| Level | Method | Reversible? | Use case |
|---|---|---|---|
| 1 | SHA-256 hash | No | Permanent de-identification |
| 2 | Random string | No | One-off anonymization |
| 3 | Deterministic token | **Yes** | Pseudonymization with reversal |
| 4 | Generalization | No | k-anonymity |
| 5 | Suppression (NULL) | No | Full removal |

In [ ]:
from zipminator.crypto.anonymization import AnonymizationEngine

engine = AnonymizationEngine()

df_tokenized = engine.apply_anonymization(
    df.copy(),
    columns=['full_name', 'email'],
    level=3,
)

print("TOKENIZED DATA (safe to share with analysts):")
print(df_tokenized[['customer_id', 'full_name', 'email']].to_string(index=False))

print("\n" + "=" * 60)
print("TOKEN MAP (keep secret, enables reversal):")
print("=" * 60)
for col, mapping in engine._token_maps.items():
    print(f"\n  Column: {col}")
    for original, token in mapping.items():
        print(f"    {original:25s} -> {token}")

print("\n" + "=" * 60)
print("REVERSED DATA (original values restored):")
print("=" * 60)
df_reversed = df_tokenized.copy()
for col, mapping in engine._token_maps.items():
    reverse_map = {token: original for original, token in mapping.items()}
    df_reversed[col] = df_reversed[col].map(reverse_map)

print(df_reversed[['customer_id', 'full_name', 'email']].to_string(index=False))

names_match = list(df_reversed['full_name']) == list(df['full_name'])
emails_match = list(df_reversed['email']) == list(df['email'])
print(f"\nReversal check - names: {'PASS' if names_match else 'FAIL'}, emails: {'PASS' if emails_match else 'FAIL'}")

In [ ]:
# ── Heatmap: File Size vs Encryption Time ──
np.random.seed(42)

file_types = ["CSV", "JSON", "Parquet", "Excel", "SQLite"]
size_labels = ["10 KB", "50 KB", "100 KB", "500 KB", "1 MB"]

base_times = np.array([
    [2.1, 8.5, 15.3, 72.1, 140.5],
    [2.8, 11.2, 19.8, 89.3, 175.2],
    [1.5, 6.2, 11.1, 51.8, 98.7],
    [3.2, 13.1, 23.5, 105.2, 210.8],
    [1.8, 7.4, 13.2, 60.5, 118.3],
])
times = base_times + np.random.normal(0, 1, base_times.shape)
times = np.clip(times, 0.5, None)

fig = zm_heatmap(
    z=times, x=size_labels, y=file_types,
    title="Encryption Time (ms) by File Type and Size",
)
fig.update_traces(
    text=np.round(times, 1),
    texttemplate="%{text} ms",
    textfont=dict(size=11, color="white"),
)
fig.update_layout(xaxis_title="File Size", yaxis_title="File Format", height=400)
fig.show()

## Step 6: Self-Destruct Timer (Optional)

For data with strict retention limits (e.g., GDPR right to erasure, internal policy),
Zipminator can automatically destroy files after a set time.

The destruction uses **DoD 5220.22-M 3-pass overwrite** before deletion:
1. Overwrite with zeros
2. Overwrite with ones
3. Overwrite with random data
4. Delete the file

This makes forensic recovery impractical on magnetic and solid-state media.

In [ ]:
from zipminator.crypto.self_destruct import SelfDestruct

sd = SelfDestruct(log_operations=True)

doomed_path = os.path.join(WORK_DIR, 'sensitive_export.csv')
df.to_csv(doomed_path, index=False)
print(f"Created sensitive file: {os.path.basename(doomed_path)}")
print(f"File size: {os.path.getsize(doomed_path):,} bytes")
print(f"File exists: {os.path.exists(doomed_path)}")

destroyed = sd.secure_delete_file(doomed_path, overwrite_passes=3)
print(f"\nSecure deletion result: {'SUCCESS' if destroyed else 'FAILED'}")
print(f"File exists after deletion: {os.path.exists(doomed_path)}")

if sd.operations_log:
    print("\nOperation log:")
    for entry in sd.operations_log:
        print(f"  {entry}")

## Summary: Protection Methods at a Glance

| Method | API | Reversible? | Quantum-safe? | Use case |
|---|---|---|---|---|
| **Password vault** | `Zipndel(password=...)` | Yes (with password) | No (AES-256, classically secure) | Daily analyst workflow |
| **PQC vault** | `Zipndel(public_key_file=...)` | Yes (with secret key) | Yes (ML-KEM-768 / FIPS 203) | Long-term archives, compliance |
| **SHA-256 masking** | `mask_columns=[...]` | No | N/A (one-way hash) | Permanent de-identification |
| **Random replacement** | `anonymize_columns=[...]` | No | N/A | One-off anonymization |
| **L3 tokenization** | `AnonymizationEngine(level=3)` | Yes (with token map) | N/A | Pseudonymization for analysis |
| **Self-destruct** | `SelfDestruct` or `self_destruct_enabled=True` | N/A (data destroyed) | N/A | GDPR erasure, retention limits |

### Recommended workflow for GDPR + DORA compliance

1. **Scan** every DataFrame before storage (`PIIScanner.scan_dataframe`)
2. **Mask** direct identifiers you do not need for analysis (`mask_columns`)
3. **Encrypt** with PQC vault for anything stored longer than 30 days
4. **Set self-destruct** timers matching your data retention policy
5. **Enable audit trail** (`audit_trail=True`) for regulatory proof

All of these steps compose into a single `Zipndel` call.

In [ ]:
shutil.rmtree(WORK_DIR, ignore_errors=True)
print(f"Cleaned up temporary directory: {WORK_DIR}")